In [1]:
import numpy as np
import ldpc.codes
import ldpc.code_util

n = 5  # specifies the lenght of the repetition code
H = ldpc.codes.rep_code(n)  # returns the repetition code parity check matrix
print(H)

<Compressed Sparse Row sparse matrix of dtype 'uint8'
	with 8 stored elements and shape (4, 5)>
  Coords	Values
  (0, 0)	1
  (0, 1)	1
  (1, 1)	1
  (1, 2)	1
  (2, 2)	1
  (2, 3)	1
  (3, 3)	1
  (3, 4)	1


In [2]:
print(H.toarray())

[[1 1 0 0 0]
 [0 1 1 0 0]
 [0 0 1 1 0]
 [0 0 0 1 1]]


In [3]:
H = ldpc.codes.hamming_code(3)
print(H.toarray())

[[0 0 0 1 1 1 1]
 [0 1 1 0 0 1 1]
 [1 0 1 0 1 0 1]]


In [4]:
n = H.shape[1]
print(f"Number of physical bits, n = {n}")

Number of physical bits, n = 7


In [5]:
k = ldpc.code_util.compute_code_dimension(H)
print(f"Number of logical bits, k = {k}")

Number of logical bits, k = 4


In [6]:
d, number_code_words_sampled, lowest_weight_codewords = (
    ldpc.code_util.estimate_code_distance(H, timeout_seconds=0.1)
)
print(
    f"Code distance estimate, d <= {d} (no. codewords sampled: {number_code_words_sampled})"
)

Code distance estimate, d <= 3 (no. codewords sampled: 427797)


In [7]:
H = ldpc.codes.hamming_code(5)  # Rank 5 Hamming Code
n, k, d_estimate = ldpc.code_util.compute_code_parameters(H)

print(f"Code parameters: [n = {n}, k = {k}, d <= {d_estimate}]")

Code parameters: [n = 31, k = 26, d <= 3]


In [8]:
import ldpc.code_util

H = np.array(
    [
        [0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0],
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1],
        [1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0],
        [1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0],
        [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0],
    ]
)

n, k, d_estimate = ldpc.code_util.compute_code_parameters(H)
print(f"Code parameters: [n = {n}, k = {k}, d <= {d_estimate}]")

Code parameters: [n = 14, k = 5, d <= 5]


In [9]:
	
import numpy as np
import ldpc.codes
import ldpc.code_util

H = ldpc.codes.hamming_code(3)
G = ldpc.code_util.construct_generator_matrix(H)
G.toarray()

array([[1, 1, 1, 0, 0, 0, 0],
       [0, 1, 1, 1, 1, 0, 0],
       [0, 1, 0, 1, 0, 1, 0],
       [0, 0, 1, 1, 0, 0, 1]], dtype=uint8)

In [10]:
temp = H @ G.T
temp.data = temp.data % 2
temp.toarray()

array([[0, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 0, 0]], dtype=uint8)

In [11]:
b = np.array([1, 0, 1, 1])

c = G.T @ b % 2
c

array([1, 0, 0, 0, 0, 1, 1])

In [12]:
H @ c % 2

array([0, 0, 0])

In [13]:
import numpy as np
import ldpc.codes
from ldpc import BpDecoder

H = ldpc.codes.rep_code(3)  # parity check matrix for the length-3 repetition code
n = H.shape[1]  # the codeword length

bpd = BpDecoder(
    H,  # the parity check matrix
    error_rate=0.1,  # the error rate on each bit
    max_iter=n,  # the maximum iteration depth for BP
    bp_method="product_sum",  # BP method. The other option is `minimum_sum'
)

In [14]:
codeword = np.array([1, 1, 1])

In [15]:
received_vector = np.array([0, 1, 1])

In [16]:
decoded_codeword = bpd.decode(received_vector)

print(decoded_codeword)

[1 1 1]


In [20]:
H = ldpc.codes.rep_code(3)  # parity check matrix for the length-3 repetition code
print(H.toarray())
n = H.shape[1]  # the codeword length
print(n)

bpd = BpDecoder(
    H,  # the parity check matrix
    error_rate=0.1,  # the error rate on each bit
    max_iter=n,  # the maximum iteration depth for BP
    bp_method="product_sum",  # BP method. The other option is `minimum_sum'
)

error = np.array([0, 1, 0])
syndrome = H @ error % 2  # the syndrome of the error

decoding = bpd.decode(syndrome)

print(f"Error: {error}")
print(f"Syndrome: {syndrome}")
print(f"Decoding: {decoding}")

[[1 1 0]
 [0 1 1]]
3
Error: [0 1 0]
Syndrome: [1 1]
Decoding: [0 1 0]


In [18]:
bpd = BpDecoder(
    H,
    max_iter=n,
    bp_method="product_sum",
    error_channel=[
        0.1,
        0,
        0.1,
    ],  # channel probability probabilities. Will overide error rate.
)

error = np.array([1, 0, 1])
syndrome = H @ error % 2
decoding = bpd.decode(syndrome)
print(f"Error: {error}")
print(f"Syndrome: {syndrome}")
print(f"Decoding: {decoding}")

Error: [1 0 1]
Syndrome: [1 1]
Decoding: [1 0 1]


In [21]:
from sympy.abc import x, y

from qldpc import codes

In [22]:
# construct the second-to-last code in Table 3 of arXiv:2308.07915v2, with code parameters [n, k, d] = [360, 12, <=24]
orders = {x: 30, y: 6}
poly_a = x**9 + y + y**2
poly_b = y**3 + x**25 + x**26
code = codes.BBCode(orders, poly_a, poly_b)

print(code)
print()
print("number of logical qubits:", code.dimension)

BBCode on 360 qubits with cyclic group orders {x: 30, y: 6} and generating polynomials
  A = x**9 + y**2 + y
  B = x**26 + x**25 + y**3

number of logical qubits: 12


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [ ]:
from codes import 